# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [5]:
df_trips.printSchema()
print("Number of rows:", df_trips.count())
print("Number of columns:", len(df_trips.columns))
df_trips.columns

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)

Number of rows: 7696617
Number of columns: 19


['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee']

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- 1) Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- 2) Which trip has the highest passanger count
- 3) What is the Average passanger count
- 4) Shortest/longest trip by distance? by time?.
- 5) busiest day/slowest single day
- 6) busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- 7) On average which day of the week is slowest/busiest
- 8) Does trip distance or num passangers affect tip amount
- 9) What was the highest "extra" charge and which trip
- 10) Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

**1)**

In [6]:
from pyspark.sql import functions as F
df_trips = df_trips.withColumn(
    "trip_id",
    F.monotonically_increasing_id()
)
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance"
).show(10)

+-----------+--------------------+---------------------+---------------+-------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|
+-----------+--------------------+---------------------+---------------+-------------+
|25769803776| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|
|25769803777| 2019-01-01 00:59:47|  2019-01-01 01:18:59|            1.0|          2.6|
|25769803778| 2018-12-21 13:48:30|  2018-12-21 13:52:40|            3.0|          0.0|
|25769803779| 2018-11-28 15:52:25|  2018-11-28 15:55:45|            5.0|          0.0|
|25769803780| 2018-11-28 15:56:57|  2018-11-28 15:58:33|            5.0|          0.0|
|25769803781| 2018-11-28 16:25:49|  2018-11-28 16:28:26|            5.0|          0.0|
|25769803782| 2018-11-28 16:29:37|  2018-11-28 16:33:43|            5.0|          0.0|
|25769803783| 2019-01-01 00:21:28|  2019-01-01 00:28:37|            1.0|          1.3|
|25769803784| 2019-01-01 00:32:01|  2019-01

**2)**

In [7]:
df_trips.orderBy(
    F.col("passenger_count").desc()
).select(
    "trip_id",
    "passenger_count",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance"
).show(10)

df_trips.select(
    F.max("passenger_count").alias("max_passenger_count")
).show()

+-----------+---------------+--------------------+---------------------+-------------+
|    trip_id|passenger_count|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|
+-----------+---------------+--------------------+---------------------+-------------+
|25770753732|            9.0| 2019-01-05 13:12:29|  2019-01-05 13:12:32|          0.0|
|25772687771|            9.0| 2019-01-13 04:13:24|  2019-01-13 04:14:34|          0.0|
|25771100063|            9.0| 2019-01-07 03:19:36|  2019-01-07 03:20:01|          0.0|
|25771815874|            9.0| 2019-01-10 00:43:10|  2019-01-10 00:43:14|          0.0|
|25774338483|            9.0| 2019-01-19 16:45:25|  2019-01-19 16:45:27|          0.0|
|25774656001|            9.0| 2019-01-21 03:46:51|  2019-01-21 03:46:56|          0.0|
|25774801566|            9.0| 2019-01-21 19:20:28|  2019-01-21 19:20:31|          0.0|
|25777090459|            9.0| 2019-01-30 18:34:12|  2019-01-30 18:34:16|          0.0|
|25777177652|            9.0| 2019-01-30 22

**3)**

In [8]:
df_trips.select(
    F.avg("passenger_count").alias("AVG Passenger Count")
).show()

+-------------------+
|AVG Passenger Count|
+-------------------+
| 1.5670317144945614|
+-------------------+



**4)**

## Distance

### Shortest

In [9]:
df_trips.orderBy(
    F.col("trip_distance").desc()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance"
).show(1)
    

+-----------+--------------------+---------------------+-------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|
+-----------+--------------------+---------------------+-------------+
|25775877867| 2019-01-25 21:56:39|  2019-01-25 22:06:08|        831.8|
+-----------+--------------------+---------------------+-------------+
only showing top 1 row


### Longest

In [10]:
df_trips.orderBy(
    F.col("trip_distance").asc()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance"
).show(1)
    

+-----------+--------------------+---------------------+-------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|
+-----------+--------------------+---------------------+-------------+
|25769803778| 2018-12-21 13:48:30|  2018-12-21 13:52:40|          0.0|
+-----------+--------------------+---------------------+-------------+
only showing top 1 row


## Time

In [11]:
df_trips = df_trips.withColumn(
    "trip_duration_seconds",
    F.unix_timestamp("tpep_dropoff_datetime")
    - F.unix_timestamp("tpep_pickup_datetime")
)

In [12]:
df_trips = df_trips.withColumn(
    "trip_duration_minutes",
    F.col("trip_duration_seconds") / 60
)

In [13]:
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).show(10)

+-----------+--------------------+---------------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+-----------+--------------------+---------------------+---------------------+
|25769803776| 2019-01-01 00:46:40|  2019-01-01 00:53:20|    6.666666666666667|
|25769803777| 2019-01-01 00:59:47|  2019-01-01 01:18:59|                 19.2|
|25769803778| 2018-12-21 13:48:30|  2018-12-21 13:52:40|    4.166666666666667|
|25769803779| 2018-11-28 15:52:25|  2018-11-28 15:55:45|   3.3333333333333335|
|25769803780| 2018-11-28 15:56:57|  2018-11-28 15:58:33|                  1.6|
|25769803781| 2018-11-28 16:25:49|  2018-11-28 16:28:26|   2.6166666666666667|
|25769803782| 2018-11-28 16:29:37|  2018-11-28 16:33:43|                  4.1|
|25769803783| 2019-01-01 00:21:28|  2019-01-01 00:28:37|                 7.15|
|25769803784| 2019-01-01 00:32:01|  2019-01-01 00:45:39|   13.633333333333333|
|25769803785| 2019-01-01 00:57:32|  2019-01-01 01:09

### Shortest

In [14]:
df_trips.orderBy(
    F.col("trip_duration_seconds").asc()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "trip_duration_seconds",
    "trip_duration_minutes"
).show(1)

+-----------+--------------------+---------------------+-------------+---------------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_seconds|trip_duration_minutes|
+-----------+--------------------+---------------------+-------------+---------------------+---------------------+
|25771006960| 2019-01-06 15:15:08|  2018-11-09 02:34:38|          3.3|             -5056830|             -84280.5|
+-----------+--------------------+---------------------+-------------+---------------------+---------------------+
only showing top 1 row


### Longest

In [15]:
df_trips.orderBy(
    F.col("trip_duration_seconds").desc()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "trip_duration_seconds",
    "trip_duration_minutes"
).show(1)

+-----------+--------------------+---------------------+-------------+---------------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_seconds|trip_duration_minutes|
+-----------+--------------------+---------------------+-------------+---------------------+---------------------+
|25769872043| 2019-01-01 07:01:20|  2019-01-31 14:29:21|          1.2|              2618881|    43648.01666666667|
+-----------+--------------------+---------------------+-------------+---------------------+---------------------+
only showing top 1 row


**5)**

In [17]:
df_trips = df_trips.withColumn(
    "pickup_date",
    F.to_date("tpep_pickup_datetime")
)
df_trips.select(
    "tpep_pickup_datetime",
    "pickup_date"
).show(10)

+--------------------+-----------+
|tpep_pickup_datetime|pickup_date|
+--------------------+-----------+
| 2019-01-01 00:46:40| 2019-01-01|
| 2019-01-01 00:59:47| 2019-01-01|
| 2018-12-21 13:48:30| 2018-12-21|
| 2018-11-28 15:52:25| 2018-11-28|
| 2018-11-28 15:56:57| 2018-11-28|
| 2018-11-28 16:25:49| 2018-11-28|
| 2018-11-28 16:29:37| 2018-11-28|
| 2019-01-01 00:21:28| 2019-01-01|
| 2019-01-01 00:32:01| 2019-01-01|
| 2019-01-01 00:57:32| 2019-01-01|
+--------------------+-----------+
only showing top 10 rows


In [19]:
df_jan = df_trips.filter(
    (F.col("pickup_date") >= "2019-01-01") &
    (F.col("pickup_date") <= "2019-01-31")
)
df_jan.select(
    F.min("pickup_date").alias("first_date"),
    F.max("pickup_date").alias("last_date")
).show()

+----------+----------+
|first_date| last_date|
+----------+----------+
|2019-01-01|2019-01-31|
+----------+----------+



In [21]:
trips_per_day = (
    df_jan
    .groupBy("pickup_date")
    .count()
)
trips_per_day.orderBy("pickup_date").show(31)

+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-01|189432|
| 2019-01-02|198737|
| 2019-01-03|223965|
| 2019-01-04|236089|
| 2019-01-05|236506|
| 2019-01-06|208823|
| 2019-01-07|228816|
| 2019-01-08|237310|
| 2019-01-09|255868|
| 2019-01-10|281863|
| 2019-01-11|291714|
| 2019-01-12|265115|
| 2019-01-13|227502|
| 2019-01-14|245082|
| 2019-01-15|267370|
| 2019-01-16|272699|
| 2019-01-17|284580|
| 2019-01-18|266848|
| 2019-01-19|236365|
| 2019-01-20|203114|
| 2019-01-21|192826|
| 2019-01-22|255178|
| 2019-01-23|261151|
| 2019-01-24|281959|
| 2019-01-25|292499|
| 2019-01-26|271993|
| 2019-01-27|220451|
| 2019-01-28|241040|
| 2019-01-29|259786|
| 2019-01-30|276774|
| 2019-01-31|284625|
+-----------+------+



In [ ]:
trips_per_day = trips_per_day.withColumn(
    "day_name",
    F.date_format("pickup_date", "EEEE")
)

### Busiest day

In [26]:
trips_per_day.orderBy(
    F.col("count").desc()
).show(5)

+-----------+------+--------+
|pickup_date| count|day_name|
+-----------+------+--------+
| 2019-01-25|292499|  Friday|
| 2019-01-11|291714|  Friday|
| 2019-01-31|284625|Thursday|
| 2019-01-17|284580|Thursday|
| 2019-01-24|281959|Thursday|
+-----------+------+--------+
only showing top 5 rows


### Slowest day

In [27]:
trips_per_day.orderBy(
    F.col("count").asc()
).show(5)

+-----------+------+---------+
|pickup_date| count| day_name|
+-----------+------+---------+
| 2019-01-01|189432|  Tuesday|
| 2019-01-21|192826|   Monday|
| 2019-01-02|198737|Wednesday|
| 2019-01-20|203114|   Sunday|
| 2019-01-06|208823|   Sunday|
+-----------+------+---------+
only showing top 5 rows


**6)**

In [31]:
df_jan = df_jan.withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime")
)
df_jan.select(
    "tpep_pickup_datetime",
    "pickup_hour"
).show(10)

+--------------------+-----------+
|tpep_pickup_datetime|pickup_hour|
+--------------------+-----------+
| 2019-01-01 00:46:40|          0|
| 2019-01-01 00:59:47|          0|
| 2019-01-01 00:21:28|          0|
| 2019-01-01 00:32:01|          0|
| 2019-01-01 00:57:32|          0|
| 2019-01-01 00:24:04|          0|
| 2019-01-01 00:21:59|          0|
| 2019-01-01 00:45:21|          0|
| 2019-01-01 00:43:19|          0|
| 2019-01-01 00:58:24|          0|
+--------------------+-----------+
only showing top 10 rows


In [33]:
trips_per_hour = (
    df_jan
    .groupBy("pickup_hour")
    .count()
)
trips_per_hour.orderBy(
    "pickup_hour"
).show(24)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0|207758|
|          1|149242|
|          2|109413|
|          3| 78084|
|          4| 61423|
|          5| 75532|
|          6|178598|
|          7|304858|
|          8|373735|
|          9|365924|
|         10|361382|
|         11|375438|
|         12|401172|
|         13|404149|
|         14|433115|
|         15|452679|
|         16|420806|
|         17|468407|
|         18|515374|
|         19|475152|
|         20|423128|
|         21|409873|
|         22|369026|
|         23|281812|
+-----------+------+



### Busiest time of day

In [34]:
trips_per_hour.orderBy(
    F.col("count").desc()
).show(5)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515374|
|         19|475152|
|         17|468407|
|         15|452679|
|         14|433115|
+-----------+------+
only showing top 5 rows


### Slowest time of day

In [35]:
trips_per_hour.orderBy(
    F.col("count").asc()
).show(5)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          4| 61423|
|          5| 75532|
|          3| 78084|
|          2|109413|
|          1|149242|
+-----------+------+
only showing top 5 rows


**7)**

In [37]:
avg_trips_by_weekday = (
    trips_per_day
    .groupBy("day_name")
    .agg(
        F.avg("count").alias("avg_trips")
    )
)
avg_trips_by_weekday.orderBy(
    F.col("avg_trips").desc()
).show()

+---------+---------+
| day_name|avg_trips|
+---------+---------+
|   Friday| 271787.5|
| Thursday| 271398.4|
|Wednesday| 253045.8|
| Saturday|252494.75|
|  Tuesday| 241815.2|
|   Monday| 226941.0|
|   Sunday| 214972.5|
+---------+---------+



### Busiest day of week on average

In [38]:
avg_trips_by_weekday.orderBy(
    F.col("avg_trips").desc()
).show(1)

+--------+---------+
|day_name|avg_trips|
+--------+---------+
|  Friday| 271787.5|
+--------+---------+
only showing top 1 row


### Slowest day of week on average

In [39]:
avg_trips_by_weekday.orderBy(
    F.col("avg_trips").asc()
).show(1)

+--------+---------+
|day_name|avg_trips|
+--------+---------+
|  Sunday| 214972.5|
+--------+---------+
only showing top 1 row


**8)**

### Distance vs Tip amount

In [40]:
distance_tip_corr = df_jan.stat.corr(
    "trip_distance",
    "tip_amount"
)

print("Correlation between trip distance and tip amount:", distance_tip_corr)

Correlation between trip distance and tip amount: 0.5269460290274736


In [41]:
df_distance = df_jan.withColumn(
    "distance_group",
    F.when(F.col("trip_distance") < 1, "0-1 miles")
     .when(F.col("trip_distance") < 2, "1-2 miles")
     .when(F.col("trip_distance") < 5, "2-5 miles")
     .when(F.col("trip_distance") < 10, "5-10 miles")
     .otherwise("10+ miles")
)

In [43]:
distance_tip_analysis = (
    df_distance
    .groupBy("distance_group")
    .agg(
        F.avg("tip_amount").alias("avg_tip"),
        F.count("*").alias("trip_count")
    )
)
distance_tip_analysis.show()

+--------------+------------------+----------+
|distance_group|           avg_tip|trip_count|
+--------------+------------------+----------+
|    5-10 miles|3.4700431561346394|    585780|
|     2-5 miles|1.9824307604593454|   1918918|
|     1-2 miles|1.3091200129885057|   2617675|
|     0-1 miles|0.9403490888599677|   2132437|
|     10+ miles| 6.219591066692631|    441270|
+--------------+------------------+----------+



### Passenger vs Tip amount

In [44]:
passenger_tip_analysis = (
    df_jan
    .groupBy("passenger_count")
    .agg(
        F.avg("tip_amount").alias("avg_tip"),
        F.count("*").alias("trip_count")
    )
    .orderBy("passenger_count")
)
passenger_tip_analysis.show()

+---------------+--------------------+----------+
|passenger_count|             avg_tip|trip_count|
+---------------+--------------------+----------+
|           NULL|0.061789899553571406|     28672|
|            0.0|  1.7869007761051638|    117381|
|            1.0|  1.8283797977982486|   5456177|
|            2.0|  1.8339551479007163|   1113794|
|            3.0|  1.7956389567716011|    314677|
|            4.0|  1.7023888221793764|    140743|
|            5.0|  1.8699418452025662|    323791|
|            6.0|  1.8567878060442715|    200788|
|            7.0|   6.542631578947368|        19|
|            8.0|   6.480689655172414|        29|
|            9.0|  3.1166666666666667|         9|
+---------------+--------------------+----------+



In [45]:
passenger_tip_corr = df_jan.stat.corr(
    "passenger_count",
    "tip_amount"
)

print(
    "Correlation between passenger count and tip amount:",
    passenger_tip_corr
)

Correlation between passenger count and tip amount: 0.004426230205403716


**9)**

In [46]:
df_jan.orderBy(
    F.col("extra").desc()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "extra",
    "total_amount"
).show(10)

+-----------+--------------------+---------------------+-------------+---------------+-----------+------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|passenger_count|fare_amount| extra|total_amount|
+-----------+--------------------+---------------------+-------------+---------------+-----------+------+------------+
|25775127259| 2019-01-23 08:58:09|  2019-01-23 08:58:09|          0.0|            1.0|  355676.98|535.38|   356214.78|
|25777257006| 2019-01-31 10:06:09|  2019-01-31 10:06:09|          0.0|            1.0|        4.5| 23.04|       28.34|
|25770346979| 2019-01-03 18:32:36|  2019-01-03 19:45:29|         16.6|            1.0|       61.0|  18.5|        92.3|
|25770114828| 2019-01-02 16:33:28|  2019-01-02 17:17:08|        17.23|            1.0|       52.0|  18.5|        88.3|
|25770352084| 2019-01-03 18:19:33|  2019-01-03 18:55:54|        16.74|            1.0|       47.5|  18.5|       94.56|
|25772258862| 2019-01-11 16:08:48|  2019-01-11 1

In [47]:
df_jan.orderBy(
    F.col("extra").desc()
).select(
    "trip_id",
    "extra"
).show(1)

+-----------+------+
|    trip_id| extra|
+-----------+------+
|25775127259|535.38|
+-----------+------+
only showing top 1 row


**10)**

### Extrem value

In [48]:
df_jan.select(
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "extra",
    "tip_amount",
    "total_amount"
).summary(
    "count",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show()

+-------+-------------+---------------------+-----------+-------+----------+------------+
|summary|trip_distance|trip_duration_minutes|fare_amount|  extra|tip_amount|total_amount|
+-------+-------------+---------------------+-----------+-------+----------+------------+
|  count|      7696080|              7696080|    7696080|7696080|   7696080|     7696080|
|    min|          0.0|             -84280.5|     -362.0|  -60.0|     -63.5|      -362.8|
|    25%|          0.9|                  6.1|        6.0|    0.0|       0.0|         8.3|
|    50%|         1.54|   10.183333333333334|        9.0|    0.0|       1.4|        11.3|
|    75%|         2.83|                16.65|       13.5|    0.5|      2.32|        16.6|
|    max|        831.8|    43648.01666666667|  623259.86| 535.38|    787.25|   623261.66|
+-------+-------------+---------------------+-----------+-------+----------+------------+



### Impossible time of trip

In [52]:
df_jan.filter(
    F.col("trip_duration_seconds") <= 0
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "trip_duration_minutes"
).show()

+-----------+--------------------+---------------------+-------------+---------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|trip_duration_minutes|
+-----------+--------------------+---------------------+-------------+---------------------+
|25769803804| 2019-01-01 00:32:59|  2019-01-01 00:32:59|          0.0|                  0.0|
|25769804457| 2019-01-01 00:39:07|  2019-01-01 00:39:07|          0.0|                  0.0|
|25769807328| 2019-01-01 00:18:46|  2019-01-01 00:18:46|          0.0|                  0.0|
|25769807437| 2019-01-01 00:54:29|  2019-01-01 00:54:29|          0.0|                  0.0|
|25769808349| 2019-01-01 00:58:14|  2019-01-01 00:58:14|          0.0|                  0.0|
|25769810249| 2019-01-01 00:18:15|  2019-01-01 00:18:15|          0.0|                  0.0|
|25769810250| 2019-01-01 00:18:17|  2019-01-01 00:18:17|          0.0|                  0.0|
|25769812271| 2019-01-01 00:27:16|  2019-01-01 00:27:16|          0.0|

In [53]:
invalid_duration_count = df_jan.filter(
    F.col("trip_duration_seconds") <= 0
).count()

print("Trips with zero or negative duration:", invalid_duration_count)

Trips with zero or negative duration: 6554


### Strange distance

In [61]:
df_jan.filter(
    F.col("trip_distance") <= 0
).select(
    "trip_id",
    "trip_distance",
    "fare_amount",
    "total_amount"
).show(10)

+-----------+-------------+-----------+------------+
|    trip_id|trip_distance|fare_amount|total_amount|
+-----------+-------------+-----------+------------+
|25769803804|          0.0|        6.5|         7.8|
|25769804235|          0.0|        3.0|         4.3|
|25769804457|          0.0|        2.5|         3.8|
|25769804621|          0.0|        3.5|         4.8|
|25769804723|          0.0|        2.5|        11.8|
|25769805114|          0.0|        3.5|         4.8|
|25769805314|          0.0|        3.0|         4.3|
|25769805411|          0.0|        7.0|        9.95|
|25769805412|          0.0|        6.5|         7.8|
|25769805413|          0.0|        8.0|         9.3|
+-----------+-------------+-----------+------------+
only showing top 10 rows


In [55]:
zero_distance_count = df_jan.filter(
    F.col("trip_distance") == 0
).count()

print("Trips with zero distance:", zero_distance_count)

Trips with zero distance: 55060


### Extrem amount

In [62]:
df_jan.orderBy(
    F.col("fare_amount").desc()
).select(
    "trip_id",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "extra",
    "tip_amount",
    "total_amount"
).show(10)

+-----------+-------------+---------------------+-----------+------+----------+------------+
|    trip_id|trip_distance|trip_duration_minutes|fare_amount| extra|tip_amount|total_amount|
+-----------+-------------+---------------------+-----------+------+----------+------------+
|25772303431|          2.4|                 19.9|  623259.86|   1.0|       0.0|   623261.66|
|25775127259|          0.0|                  0.0|  355676.98|535.38|       0.0|   356214.78|
|25771963747|          0.0|                  0.0|    36090.3|   0.0|       0.0|     36090.3|
|25771696557|          0.0|                  0.0|   34674.65|   0.0|       0.0|    34674.65|
|25771453227|          0.0|                  0.0|   33023.53|   0.0|       0.0|    33023.53|
|25770455409|          0.0|                  0.0|   31107.91|   0.0|       0.0|    31107.91|
|25772248553|          0.0|                  0.0|   30444.52|   0.0|       0.0|    30444.52|
|25770541107|          0.0|                  0.0|   30130.71|   0.0|  

In [63]:
df_jan.orderBy(
    F.col("total_amount").desc()
).select(
    "trip_id",
    "trip_distance",
    "fare_amount",
    "extra",
    "tip_amount",
    "total_amount"
).show(10)

+-----------+-------------+-----------+------+----------+------------+
|    trip_id|trip_distance|fare_amount| extra|tip_amount|total_amount|
+-----------+-------------+-----------+------+----------+------------+
|25772303431|          2.4|  623259.86|   1.0|       0.0|   623261.66|
|25775127259|          0.0|  355676.98|535.38|       0.0|   356214.78|
|25771963747|          0.0|    36090.3|   0.0|       0.0|     36090.3|
|25771696557|          0.0|   34674.65|   0.0|       0.0|    34674.65|
|25771453227|          0.0|   33023.53|   0.0|       0.0|    33023.53|
|25770455409|          0.0|   31107.91|   0.0|       0.0|    31107.91|
|25772248553|          0.0|   30444.52|   0.0|       0.0|    30444.52|
|25770541107|          0.0|   30130.71|   0.0|       0.0|    30130.71|
|25771223461|          0.0|   25628.96|   0.0|       0.0|    25628.96|
|25770104888|          0.0|   25356.38|   0.0|       0.0|    25356.38|
+-----------+-------------+-----------+------+----------+------------+
only s

In [64]:
df_jan.filter(
    (F.col("fare_amount") < 0) |
    (F.col("total_amount") < 0)
).select(
    "trip_id",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount"
).show(20)

+-----------+-------------+-----------+----------+------------+
|    trip_id|trip_distance|fare_amount|tip_amount|total_amount|
+-----------+-------------+-----------+----------+------------+
|25769804439|          0.1|       -2.5|       0.0|        -3.8|
|25769806178|         4.13|      -19.0|       0.0|       -20.3|
|25769806317|         1.35|       -8.5|       0.0|        -9.8|
|25769806320|          0.0|       -2.5|       0.0|        -3.8|
|25769806323|         0.16|       -3.0|       0.0|        -4.3|
|25769807004|         0.42|       -4.5|       0.0|        -5.8|
|25769807558|         0.47|      -24.0|       0.0|       -24.8|
|25769807703|          0.7|      -52.0|       0.0|       -52.8|
|25769808037|          0.4|       -4.0|       0.0|        -5.3|
|25769809076|        15.13|      -43.5|       0.0|       -44.8|
|25769809557|          0.0|       -3.5|       0.0|        -4.8|
|25769810853|         0.13|       -3.5|       0.0|        -4.8|
|25769813140|         0.36|       -3.5| 

### Part 2

- 1) Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- 2) which borough had most pickups? dropoffs?
- 3) what are the busy/slow times by borough 
- 4) what are the busiest days of the week by borough?
- 5) what is the average trip distance by borough?
- 6) what is the average trip fare by borough?
- 7) highest/lowest faire amounts for a trip, what burough is associated with the each
- 8) load the dataset from the most recently available january, is there a change to any of the average metrics.

In [65]:
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

response = requests.get(zone_url)

zone_file = "taxi_zone_lookup.csv"

if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

print("Status:", response.status_code)

Status: 200


In [67]:
df_zones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(zone_file)
)
df_zones.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


In [68]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing